#TEXT PREPROCESSING

In [ ]:
#setup

from pathlib import Path
import string
import pandas as pd

from nltk.tokenize import wordpunct_tokenize
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS


#pd.set_option("display.max_colwidth", 120)

# Path del dataset
DATA_PATH = Path("../data/Sentences_50Agree.txt")

# Cartella output specifica per Task 2
OUTPUT_DIR = Path("../results/task2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Caricamento dataset
rows = []

with open(DATA_PATH, "r", encoding="latin-1") as f:
    for line in f:
        line = line.strip()
        if line:
            text, label = line.rsplit("@", 1)
            rows.append({
                "text": text.strip(),
                "label": label.strip()
            })

df = pd.DataFrame(rows)

print("Dataset loaded successfully.")
print("Number of examples:", len(df))
print("Labels:", sorted(df["label"].unique()))

df.head()

In [ ]:
# Standard English stop words
stop_words = set(ENGLISH_STOP_WORDS)

# Words that may be important for sentiment or financial meaning
words_to_keep = {
    "no", "not", "nor",
    "up", "down",
    "increase", "decrease",
    "increased", "decreased",
    "rise", "rose", "fall", "fell",
    "profit", "loss"
}

stop_words = stop_words - words_to_keep

print("Number of stop words:", len(stop_words))

In [ ]:
#1-conservative preprocessing: lowercase and remove extra whitespace

def preprocess_minimal(text):
    text = text.lower()
    text = " ".join(text.split())
    return text

df["text_minimal"] = df["text"].apply(preprocess_minimal)

df[["text", "text_minimal"]].head()

In [ ]:
#2-remove punctuation and tokenize

def preprocess_no_punctuation(text):
    tokens = wordpunct_tokenize(text.lower())
    
    cleaned_tokens = [
        token for token in tokens
        if token not in string.punctuation
    ]
    
    return " ".join(cleaned_tokens)

df["text_no_punctuation"] = df["text"].apply(preprocess_no_punctuation)

df[["text", "text_no_punctuation"]].head()

In [ ]:
#3-remove stop words

def preprocess_stopwords(text):
    tokens = wordpunct_tokenize(text.lower())
    
    cleaned_tokens = [
        token for token in tokens
        if token not in string.punctuation
        and token not in stop_words
    ]
    
    return " ".join(cleaned_tokens)

df["text_stopwords"] = df["text"].apply(preprocess_stopwords)

df[["text", "text_stopwords"]].head()

In [ ]:
#4-stemming

stemmer = PorterStemmer()

def preprocess_stemming(text):
    tokens = wordpunct_tokenize(text.lower())
    
    cleaned_tokens = [
        stemmer.stem(token)
        for token in tokens
        if token not in string.punctuation
        and token not in stop_words
    ]
    
    return " ".join(cleaned_tokens)

df["text_stemming"] = df["text"].apply(preprocess_stemming)

df[["text", "text_stemming"]].head()

In [ ]:
comparison_examples = df[
    ["text", "text_minimal", "text_no_punctuation", "text_stopwords", "text_stemming", "label"]
].sample(5, random_state=42)

comparison_examples

In [ ]:
preprocessing_columns = [
    "text_minimal",
    "text_no_punctuation",
    "text_stopwords",
    "text_stemming"
]

summary_rows = []

for column in preprocessing_columns:
    tokenized_texts = df[column].apply(wordpunct_tokenize)
    
    all_tokens = [
        token
        for tokens in tokenized_texts
        for token in tokens
    ]
    
    vocabulary = set(all_tokens)
    
    summary_rows.append({
        "preprocessing": column,
        "avg_tokens_per_headline": round(tokenized_texts.apply(len).mean(), 2),
        "vocabulary_size": len(vocabulary),
        "total_tokens": len(all_tokens)
    })

preprocessing_summary = pd.DataFrame(summary_rows)

preprocessing_summary

In [ ]:
financial_terms = ["eur", "euro", "usd", "mn", "mln", "million", "%", "profit", "loss"]

financial_check = []

for column in preprocessing_columns:
    text_joined = " ".join(df[column])
    
    for term in financial_terms:
        financial_check.append({
            "preprocessing": column,
            "term": term,
            "appears": term in text_joined
        })

financial_check = pd.DataFrame(financial_check)

financial_check.pivot(index="term", columns="preprocessing", values="appears")

In [ ]:
selected_preprocessing = {
    "minimal": "text_minimal",
    "no_punctuation": "text_no_punctuation",
    "stopwords": "text_stopwords",
    "stemming": "text_stemming"
}

selected_preprocessing

In [ ]:
X_minimal = df["text_minimal"]
X_no_punctuation = df["text_no_punctuation"]
X_stopwords = df["text_stopwords"]
X_stemming = df["text_stemming"]
y = df["label"]

In [ ]:
task2_output_path = OUTPUT_DIR / "preprocessed_texts.csv"

df[
    ["text", "label", "text_minimal", "text_no_punctuation", "text_stopwords", "text_stemming"]
].to_csv(task2_output_path, index=False)

print("Saved preprocessed dataset to:", task2_output_path)


In [ ]:
print("TASK 2 SUMMARY")
print("=" * 60)

print("Preprocessing variants created:")
for column in preprocessing_columns:
    print("-", column)

print()
print("Quantitative comparison:")
display(preprocessing_summary)

print()
print("Example transformations:")
display(comparison_examples)